# Accuracy Test Runner

This cell runs cocotb accuracy tests for FP32 and MXINT8 addition/multiplication,
for both `subcomponents` and `combined` variants.
Each entry invokes `make -f Makefile results.xml` with variant-specific env vars.
Run this with the `prog-synth-verif` environment.


In [ ]:
import os
import subprocess
from pathlib import Path

def find_repo_root() -> Path:
    root = Path.cwd().resolve()
    while root != root.parent:
        if (root / "accuracy_tests" / "Makefile").exists():
            return root
        root = root.parent
    raise RuntimeError("Could not locate repo root containing accuracy_tests/Makefile")

ROOT = find_repo_root()
ACC_DIR = ROOT / "accuracy_tests"

TESTS = [
    {
        "name": "mxint8_adder_subcomponents",
        "soln": "solution_mxint8addition_full_sum",
        "top": "add_full_sum",
        "module": "tests.addition.test_mxint8_adder",
        "extra_env": {"MXINT8_ADD_VARIANT": "subcomponents"},
    },
    {
        "name": "mxint8_adder_combined",
        "soln": "solution_mxint8addition_full_sum_combined",
        "top": "add_full_sum",
        "module": "tests.addition.test_mxint8_adder",
        "extra_env": {"MXINT8_ADD_VARIANT": "combined"},
    },
    {
        "name": "mxint8_multiplier_subcomponents",
        "soln": "solution_mxint8multiplication_full_product",
        "top": "mult_mxint_full_product",
        "module": "tests.multiplication.test_mxint8_multiplier",
        "extra_env": {"MXINT8_MUL_VARIANT": "subcomponents"},
    },
    {
        "name": "mxint8_multiplier_combined",
        "soln": "solution_mxint8multiplication_full_product_combined",
        "top": "mult_mxint_full_product",
        "module": "tests.multiplication.test_mxint8_multiplier",
        "extra_env": {"MXINT8_MUL_VARIANT": "combined"},
    },
    {
        "name": "fp32_adder_subcomponents",
        "soln": "solution_fp32addition_fp32_full_sum",
        "top": "fp32_sum",
        "module": "tests.addition.test_fp32_adder",
        "extra_env": {"FP32_ADD_VARIANT": "subcomponents"},
    },
    {
        "name": "fp32_adder_combined",
        "soln": "solution_fp32addition_fp32_full_sum_combined",
        "top": "fp32_sum",
        "module": "tests.addition.test_fp32_adder",
        "extra_env": {"FP32_ADD_VARIANT": "combined"},
    },
    {
        "name": "fp32_multiplier_subcomponents",
        "soln": "solution_fp32multiplication_full_product",
        "top": "fp32_full_mul",
        "module": "tests.multiplication.test_fp32_multiplier",
        "extra_env": {"FP32_MUL_VARIANT": "subcomponents"},
    },
    {
        "name": "fp32_multiplier_combined",
        "soln": "solution_fp32multiplication_full_product_combined",
        "top": "fp32_full_mul",
        "module": "tests.multiplication.test_fp32_multiplier",
        "extra_env": {"FP32_MUL_VARIANT": "combined"},
    },
]

SHOW_FULL_LOG = False
TAIL_LINES = 200

def tail(text: str, lines: int) -> str:
    parts = text.splitlines()
    return "\n".join(parts[-lines:])

def extract_metrics(text: str) -> dict:
    metrics = {}
    for line in text.splitlines():
        line = line.strip()
        if line.startswith('Ran '):
            metrics['ran'] = line
        elif 'Percent Within' in line:
            metrics.setdefault('percent_within', []).append(line)
        elif 'ULP avg=' in line:
            metrics['ulp_summary'] = line
        elif 'Exact ' in line and 'ULP' in line:
            metrics['ulp_exact'] = line
        elif 'Max Absolute Error:' in line:
            metrics.setdefault('max_abs', []).append(line)
        elif 'Average Absolute Error:' in line:
            metrics.setdefault('avg_abs', []).append(line)
        elif '99th Percentile' in line:
            metrics.setdefault('p99', []).append(line)
        elif '95th Percentile' in line:
            metrics.setdefault('p95', []).append(line)
    return metrics

results = []

for t in TESTS:
    print(f"\n=== Running {t['name']} ===")

    verilog_dir = ROOT / 'results' / 'HLS' / t['soln'] / 'verilog_out'
    if not verilog_dir.exists():
        print(f"Skipping {t['name']} (missing hardware output: {verilog_dir})")
        results.append({"name": t['name'], "returncode": None, "metrics": {}})
        continue

    env = os.environ.copy()
    env.update({
        "HLS_SOLN": t["soln"],
        "TOPLEVEL": t["top"],
        "MODULE": t["module"],
    })
    env.update(t.get('extra_env', {}))

    results_xml = ACC_DIR / "results.xml"
    if results_xml.exists():
        results_xml.unlink()

    proc = subprocess.run(
        ["make", "-f", "Makefile", "results.xml"],
        cwd=ACC_DIR,
        env=env,
        text=True,
        capture_output=True,
    )

    stdout = proc.stdout or ""
    stderr = proc.stderr or ""

    if SHOW_FULL_LOG:
        print(stdout)
        if stderr:
            print(stderr)
    else:
        print(tail(stdout, TAIL_LINES))
        if stderr:
            print("\n[stderr tail]\n" + tail(stderr, TAIL_LINES))

    metrics = extract_metrics(stdout)
    if not metrics and stderr:
        metrics = extract_metrics(stderr)

    results.append({
        "name": t["name"],
        "returncode": proc.returncode,
        "metrics": metrics,
    })

print("\n=== Summary ===")
for r in results:
    if r['returncode'] is None:
        status = "SKIP"
    else:
        status = "PASS" if r["returncode"] == 0 else "FAIL"
    print(f"{r['name']}: {status}")
    m = r.get("metrics", {})
    if not m:
        continue
    if 'ran' in m:
        print(f"  {m['ran']}")
    for line in m.get('percent_within', []):
        print(f"  {line}")
    for line in m.get('max_abs', []):
        print(f"  {line}")
    for line in m.get('avg_abs', []):
        print(f"  {line}")
    for line in m.get('p99', []):
        print(f"  {line}")
    for line in m.get('p95', []):
        print(f"  {line}")
    if 'ulp_summary' in m:
        print(f"  {m['ulp_summary']}")
    if 'ulp_exact' in m:
        print(f"  {m['ulp_exact']}")


# Hardware Results

This cell runs the full pipeline for each FP32/MXINT8 add/mul variant:
SyGuS synthesis -> SMT2 export -> smt2c translation -> HLS C++ conversion -> Vitis HLS/Vivado.
It covers both `subcomponents` and `combined` variants and prints a consolidated summary.
Run this with the `prog-synth` environment.


In [1]:
import os
import sys
import subprocess
from pathlib import Path


def find_repo_root() -> Path:
    root = Path.cwd().resolve()
    while root != root.parent:
        if (root / 'src' / 'synthesis_driver.py').exists():
            return root
        root = root.parent
    raise RuntimeError('Could not locate repo root containing src/synthesis_driver.py')


ROOT = find_repo_root()

RUN_IMPL = True
ENABLE_DIRECTED_IO = True
NUM_ITERATIONS = 30
SOLVER_TIMEOUT_SECONDS = 15
SHOW_FULL_LOG = False
TAIL_LINES = 200

JOBS = [
    {"name": "mxint8_add_subcomponents", "target": "mxint8_add", "component": "full_sum"},
    {"name": "mxint8_add_combined", "target": "mxint8_add", "component": "full_sum_combined"},
    {"name": "mxint8_mul_subcomponents", "target": "mxint8_mul", "component": "full_product"},
    {"name": "mxint8_mul_combined", "target": "mxint8_mul", "component": "full_product_combined"},
    {"name": "fp32_add_subcomponents", "target": "fp32_add", "component": "fp32_full_sum"},
    {"name": "fp32_add_combined", "target": "fp32_add", "component": "fp32_full_sum_combined"},
    {"name": "fp32_mul_subcomponents", "target": "fp32_mul", "component": "full_product"},
    {"name": "fp32_mul_combined", "target": "fp32_mul", "component": "full_product_combined"},
]


def tail(text: str, lines: int) -> str:
    parts = text.splitlines()
    return '\n'.join(parts[-lines:])


print(f"Running full synthesis->verilog pipeline for {len(JOBS)} targets.")
summary = []

for job in JOBS:
    print(f"\n=== Pipeline run: {job['name']} ===")

    env = os.environ.copy()
    env.update({
        'SYNTH_TARGET': job['target'],
        'SYNTH_COMPONENT': job['component'],
        'SYNTH_RUN_IMPL': '1' if RUN_IMPL else '0',
        'SYNTH_ENABLE_DIRECTED_IO': '1' if ENABLE_DIRECTED_IO else '0',
        'SYNTH_NUM_ITERATIONS': str(NUM_ITERATIONS),
        'SYNTH_SOLVER_TIMEOUT': str(SOLVER_TIMEOUT_SECONDS),
    })

    proc = subprocess.run(
        [sys.executable, '-m', 'src.synthesis_driver'],
        cwd=ROOT,
        env=env,
        text=True,
        capture_output=True,
    )

    stdout = proc.stdout or ''
    stderr = proc.stderr or ''

    if SHOW_FULL_LOG:
        print(stdout)
        if stderr:
            print('\n[stderr]\n' + stderr)
    else:
        print(tail(stdout, TAIL_LINES))
        if stderr:
            print('\n[stderr tail]\n' + tail(stderr, TAIL_LINES))

    status = 'PASS' if proc.returncode == 0 else 'FAIL'
    summary.append((job['name'], status, proc.returncode))

print('\n=== Hardware Pipeline Summary ===')
for name, status, rc in summary:
    print(f"{name}: {status} (rc={rc})")


Running full synthesis->verilog pipeline for 8 targets.

=== Pipeline run: mxint8_add_subcomponents ===

--- Iteration 23/30 ---
Generating new constraint with inputs: (-51.3878800400866, 37.514031958806996)
(constraint (= (add_full_sum #b1001 #b0110 #b0100 #b0110) #b10100101))
SUCCESS: Constraint accepted. Total constraints: 12

--- Iteration 24/30 ---
Generating new constraint with inputs: (32.00823334015679, 45.47891900207904)
(constraint (= (add_full_sum #b0100 #b0110 #b0101 #b0110) #b01000111))
[CVC5] Solver did not return a valid solution.
[STDOUT]:
fail
SKIPPED: Constraint from (32.008, 45.479) caused timeout or error.

--- Iteration 25/30 ---
Generating new constraint with inputs: (62.18723322579538, 58.19764073512238)
(constraint (= (add_full_sum #b0111 #b0110 #b0111 #b0110) #b01110111))
SUCCESS: Constraint accepted. Total constraints: 13

--- Iteration 26/30 ---
Generating new constraint with inputs: (53.65711518455866, 9.150823791631112)
(constraint (= (add_full_sum #b0110 #